# R Smoke Test -- Snowflake Workspace Notebook

Minimal validation that R, **snowflakeR**, and **RSnowflake** work correctly
in a Snowflake Workspace Notebook using only the public GitHub repos:
- [snowflake-notebook-multilang](https://github.com/Snowflake-Labs/snowflake-notebook-multilang) (R bootstrap)
- [snowflakeR](https://github.com/Snowflake-Labs/snowflakeR) (ML platform: Registry, Feature Store)
- [RSnowflake](https://github.com/Snowflake-Labs/RSnowflake) (DBI connector)

**Before you start:** Copy `notebook_config.yaml.template` to `notebook_config.yaml`
and edit it with your warehouse, database, and schema.

**Setup:**
1. Run **Section 0** to ensure the EAI has all required domains.
2. If this is a fresh setup (no EAI attached yet), follow the printed
   instructions to attach it via the Snowsight UI, then re-run Section 0.
3. Once Section 0 reports success, run from **Section 1** onward.

**What this tests:**
1. R language execution via `%%R` magic
2. snowflakeR connection and query
3. snowflakeR Model Registry connectivity
4. snowflakeR Feature Store connectivity
5. RSnowflake DBI connectivity (SPCS OAuth -- no PAT needed)

**Estimated time:** ~1.5 min with tarballs, ~3.5 min from GitHub (~45 sec cached)

## 0. EAI Setup

Snowflake Workspace Notebooks block all outbound network traffic by default.
This cell ensures your External Access Integration (EAI) has all the domains
needed to install R packages from conda-forge, CRAN, GitHub, and PyPI.

### How it works

- **EAI already attached?** The cell introspects its network rule, adds any
  missing domains, and changes take effect immediately -- no restart needed.
- **No EAI yet?** The cell creates one and prints instructions to attach it
  via the Snowsight UI (one-time manual step).
- **No privileges?** The cell prints the complete SQL for your admin.

### EAI name resolution

1. Checks `notebook_config.yaml` for an explicit `eai.name`
2. Falls back to the convention name `multilang_notebook_eai`

### First-time setup (if no EAI exists yet)

After this cell creates the EAI, attach it to your notebook service:

1. Click **Connected** (top-left of the notebook toolbar)
2. Hover over your service name and click **Edit**
3. Scroll to **External Access**
4. Toggle **ON** the EAI > **Save**
5. Service restarts automatically
6. Re-run this cell (it will update the rule), then continue to **Section 1**

### Why can't the notebook attach the EAI itself?

Workspace Notebook services are system-managed SPCS containers. There is no
supported SQL or API to attach EAIs programmatically -- only to modify an
EAI that is already attached. The initial attachment is a one-time step via
the Snowsight service settings UI.

Once created and attached, the same EAI can be reused across multiple
Workspace Notebooks -- it is a one-time setup per service.

In [ ]:
from snowflake.snowpark.context import get_active_session
from eai_helper import ensure_eai

session = get_active_session()
result = ensure_eai(
    session,
    lang_config="r_smoke_test_config.yaml",
    notebook_config="notebook_config.yaml",
)
print(f"\nAction: {result['action']}")

## 1. Bootstrap R Environment

Installs R via micromamba + conda-forge using the
[sfnb-multilang](https://github.com/Snowflake-Labs/snowflake-notebook-multilang)
toolkit (~3 min first run, ~2 sec cached).

The config file `r_smoke_test_config.yaml` pre-installs conda packages
needed by both snowflakeR and RSnowflake so that the subsequent `pak` installs
are fast.

**Prerequisite:** Section 0 EAI must be enabled and kernel restarted.

In [ ]:
import os, json, yaml

config_path = "r_smoke_test_config.yaml"
with open(config_path) as f:
    cfg = yaml.safe_load(f)

tarballs = (cfg.get("languages", {}).get("r", {}).get("tarballs")) or {}
if tarballs:
    os.environ["R_TARBALL_CONFIG"] = json.dumps(tarballs)
    print(f"Tarball config: {len(tarballs)} package(s)")
    for pkg, src in tarballs.items():
        label = "URL" if src.startswith("http") else "local"
        print(f"  {pkg}: {label} -> {src}")

from sfnb_setup import install_r
install_r(config=config_path)

## 2. Install snowflakeR and RSnowflake

Three install methods are supported (auto-detected per package):

| Method | How | First run | Cached |
|--------|-----|-----------|--------|
| **URL download** (fast) | Set URL in `r_smoke_test_config.yaml` | ~10 sec | ~10 sec |
| **Local tarball** (fast) | Upload `.tar.gz` to Workspace or set path in config | ~10 sec | ~10 sec |
| **GitHub via pak** (fallback) | No config needed -- downloads from GitHub | ~2 min | ~10 sec |

Configure tarballs in `r_smoke_test_config.yaml` under the `tarballs` key.
Each entry can be a URL, a local file path, or omitted. URLs are downloaded
at install time (requires EAI access to the host). Omitted entries trigger
a recursive Workspace search; if not found, `pak` pulls from GitHub.

Additional R packages (beyond snowflakeR and RSnowflake) can also be
listed -- they will be installed from their URL or local tarball.

In [ ]:
%%R
options(
  repos = c(CRAN = "https://cloud.r-project.org"),
  pkg.sysreqs = FALSE
)

tarball_cfg <- tryCatch(
  jsonlite::fromJSON(Sys.getenv("R_TARBALL_CONFIG", "{}")),
  error = function(e) list()
)

resolve_tarball <- function(pkg) {
  src <- tarball_cfg[[pkg]]
  if (!is.null(src) && nzchar(src)) {
    if (grepl("^https?://", src)) {
      dest <- file.path(tempdir(), basename(src))
      cat("  Downloading", pkg, "from", src, "\n")
      download.file(src, dest, quiet = TRUE)
      return(dest)
    }
    if (file.exists(src)) return(src)
    cat("  Warning: configured path not found:", src, "\n")
  }
  regex <- paste0("^", pkg, "_.*\\.tar\\.gz$")
  hits <- list.files(".", pattern = regex, recursive = TRUE, full.names = TRUE)
  if (length(hits) == 0) return(NULL)
  if (length(hits) == 1) return(hits)
  vers <- sub("\\.tar\\.gz$", "", sub(paste0("^", pkg, "_"), "", basename(hits)))
  hits[order(numeric_version(vers, strict = FALSE), decreasing = TRUE)[1]]
}

install_from_tarball <- function(pkg) {
  tar <- resolve_tarball(pkg)
  if (is.null(tar)) return(FALSE)
  cat("  ", pkg, "<-", tar, "\n")
  install.packages(tar, repos = NULL, type = "source")
  TRUE
}

extra_pkgs <- setdiff(names(tarball_cfg), c("snowflakeR", "RSnowflake"))

sfr_ok <- install_from_tarball("snowflakeR")
rsf_ok <- install_from_tarball("RSnowflake")

if (!sfr_ok || !rsf_ok) {
  cat("Falling back to GitHub for missing packages...\n")
  if (!requireNamespace("pak", quietly = TRUE))
    install.packages("pak", type = "source", quiet = TRUE)
  if (!sfr_ok) pak::pak("Snowflake-Labs/snowflakeR", ask = FALSE, upgrade = FALSE)
  if (!rsf_ok) pak::pak("Snowflake-Labs/RSnowflake", ask = FALSE, upgrade = FALSE)
}

for (pkg in extra_pkgs) {
  if (!install_from_tarball(pkg))
    cat("  Warning: could not resolve tarball for", pkg, "\n")
}

cat("\nInstalled versions:\n")
cat("  snowflakeR ", as.character(packageVersion("snowflakeR")), "\n")
cat("  RSnowflake ", as.character(packageVersion("RSnowflake")), "\n")
for (pkg in extra_pkgs) {
  v <- tryCatch(as.character(packageVersion(pkg)), error = function(e) "NOT FOUND")
  cat(" ", pkg, v, "\n")
}

## 3. Test: R Language Execution

Verify that R code runs correctly via the `%%R` magic.

In [ ]:
%%R
cat("=== R Language Execution Test ===\n\n")

cat("R version:", R.version.string, "\n")
cat("Platform: ", R.version$platform, "\n\n")

df <- data.frame(
  x     = 1:5,
  y     = c(2.3, 4.1, 1.8, 5.7, 3.2),
  label = letters[1:5]
)
cat("Data frame created:\n")
print(df)

cat("\nSum of x:  ", sum(df$x), "\n")
cat("Mean of y: ", round(mean(df$y), 4), "\n")

cat("\n[PASS] R language execution works.\n")

## 4. Test: snowflakeR Connection & Query

Connect to Snowflake using `snowflakeR` (uses the active Workspace session
via reticulate -- no credentials needed).

In [ ]:
%%R
cat("=== snowflakeR Connection & Query Test ===\n\n")

library(snowflakeR)

conn <- sfr_connect()
conn <- sfr_load_notebook_config(conn)
conn

result <- sfr_query(
  conn,
  "SELECT CURRENT_TIMESTAMP() AS TS, CURRENT_USER() AS WHO, CURRENT_DATABASE() AS DB"
)
rprint(result)

cat("\n[PASS] snowflakeR connection and query work.\n")

## 5. Test: snowflakeR Model Registry

Connect to the Model Registry and list registered models.
An empty list is a valid result -- it confirms the Python bridge and
Snowflake ML API are working.

In [ ]:
%%R
cat("=== Model Registry Test ===\n\n")

reg <- sfr_model_registry(conn)
cat("Registry context:\n")
print(reg)

models <- sfr_show_models(reg)
cat("\nModels in registry:", nrow(models), "\n")
if (nrow(models) > 0) {
  rprint(head(models))
} else {
  cat("(empty -- expected for a fresh database)\n")
}

cat("\n[PASS] Model Registry connectivity works.\n")

## 6. Test: snowflakeR Feature Store

Connect to the Feature Store and list entities and feature views.
`create = TRUE` initializes Feature Store metadata in the schema if needed.
Empty results are valid -- they confirm API connectivity.

In [ ]:
%%R
cat("=== Feature Store Test ===\n\n")

fs <- sfr_feature_store(
  conn,
  database  = conn$database,
  schema    = conn$schema,
  warehouse = conn$warehouse,
  create    = TRUE
)
cat("Feature Store context:\n")
print(fs)

entities <- sfr_list_entities(fs)
cat("\nEntities:", nrow(entities), "\n")
if (nrow(entities) > 0) rprint(head(entities))

fvs <- sfr_list_feature_views(fs)
cat("Feature Views:", nrow(fvs), "\n")
if (nrow(fvs) > 0) rprint(head(fvs))

cat("\n[PASS] Feature Store connectivity works.\n")

## 7. Test: RSnowflake DBI Connectivity

RSnowflake connects via the Snowflake SQL API v2. In Workspace Notebooks
the built-in SPCS OAuth token is used automatically -- no PAT required.

The cell below sets environment variables that RSnowflake reads for
account, database, warehouse, and role context, then runs a round-trip
DBI test (connect, query, write, read, cleanup).

In [ ]:
import os
from snowflake.snowpark.context import get_active_session

session = get_active_session()

os.environ["SNOWFLAKE_ACCOUNT"] = session.get_current_account().replace('"', '')
os.environ["SNOWFLAKE_USER"] = session.sql("SELECT CURRENT_USER()").collect()[0][0]
os.environ["SNOWFLAKE_DATABASE"] = (session.get_current_database() or "").replace('"', '')
os.environ["SNOWFLAKE_SCHEMA"] = (session.get_current_schema() or "").replace('"', '')
os.environ["SNOWFLAKE_WAREHOUSE"] = (session.get_current_warehouse() or "").replace('"', '')
os.environ["SNOWFLAKE_ROLE"] = (session.get_current_role() or "").replace('"', '')

print(f"Account:   {os.environ['SNOWFLAKE_ACCOUNT']}")
print(f"Database:  {os.environ['SNOWFLAKE_DATABASE']}")
print(f"Warehouse: {os.environ['SNOWFLAKE_WAREHOUSE']}")
print("Auth:      SPCS OAuth token (built-in)")

In [ ]:
%%R
cat("=== RSnowflake DBI Test ===\n\n")

if (!nzchar(Sys.getenv("TZ", ""))) Sys.setenv(TZ = "UTC")

library(RSnowflake)
library(DBI)

con <- dbConnect(Snowflake())
cat("Connected:", dbIsValid(con), "\n")
cat("Host:     ", dbGetInfo(con)$host, "\n\n")

ts <- dbGetQuery(con, "SELECT CURRENT_TIMESTAMP() AS TS")
cat("Server time:", as.character(ts$TS), "\n")

test_tbl <- "R_SMOKE_TEST_DBI"
dbWriteTable(con, test_tbl, mtcars[1:5, ], overwrite = TRUE)
result <- dbReadTable(con, test_tbl)
cat("Wrote 5 rows, read back:", nrow(result), "rows\n")
stopifnot(nrow(result) == 5)

dbRemoveTable(con, test_tbl)
dbDisconnect(con)

cat("\n[PASS] RSnowflake DBI connectivity works.\n")

## 8. Summary

Recap of all test results.

In [ ]:
%%R
cat("\n")
cat("============================================================\n")
cat("  R Smoke Test -- Summary\n")
cat("============================================================\n")
cat("  R version:      ", R.version.string, "\n")
cat("  snowflakeR:     ", as.character(packageVersion("snowflakeR")), "\n")
cat("  RSnowflake:     ", as.character(packageVersion("RSnowflake")), "\n")
cat("  sfnb-multilang:  (pip-installed from GitHub)\n")
cat("------------------------------------------------------------\n")
cat("  R execution:       PASS\n")
cat("  snowflakeR query:  PASS\n")
cat("  Model Registry:    PASS\n")
cat("  Feature Store:     PASS\n")
cat("  RSnowflake DBI:    PASS\n")
cat("============================================================\n")
cat("  All public packages validated successfully.\n")
cat("============================================================\n")